In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# =====================================================
# Load Dataset
# =====================================================

df = pd.read_csv("../data/processed/dataset_labeled.csv")

# =====================================================
# Define Criteria
# =====================================================

benefit_cols = [
    "office_count_500m",
    "retail_count_500m",
    "parking_space_count_500m",
    "recreation_count_500m",
    "bank_count_500m",
    "bus_stop_count_500m",
    "college_count_500m",
    "hospital_count_500m",
    "school_count_500m",
    "cinema_count_500m",
    "museum_count_500m",
    "temple_count_500m",
    "clinic_count_500m",
    "nearest_restaurant_m"
]

cost_cols = [
    "competitor_count_500m",
    "avg_restaurant_rating_500m",
    "avg_review_ratings_500m"
]

criteria = benefit_cols + cost_cols

# =====================================================
# Convert to numeric
# =====================================================

for c in criteria:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df[criteria] = df[criteria].fillna(df[criteria].median())

# =====================================================
# Min-Max Normalization
# =====================================================

normalized = pd.DataFrame(index=df.index)

for col in benefit_cols:

    mn = df[col].min()
    mx = df[col].max()

    if mx == mn:
        normalized[col] = 1
    else:
        normalized[col] = (df[col] - mn) / (mx - mn)

for col in cost_cols:

    mn = df[col].min()
    mx = df[col].max()

    if mx == mn:
        normalized[col] = 1
    else:
        normalized[col] = (mx - df[col]) / (mx - mn)

# =====================================================
# Entropy Weight Method
# =====================================================

eps = 1e-12

P = normalized.div(normalized.sum(axis=0), axis=1)
P = P.replace(0, eps)

n = len(df)

entropy = -(P * np.log(P)).sum(axis=0) / np.log(n)

divergence = 1 - entropy

weights = divergence / divergence.sum()

# =====================================================
# Save Weight Table
# =====================================================

weight_table = pd.DataFrame({
    "feature": criteria,
    "criterion_type": ["Benefit"]*len(benefit_cols)+["Cost"]*len(cost_cols),
    "entropy": entropy.values,
    "divergence": divergence.values,
    "weight": weights.values
})

weight_table = weight_table.sort_values(
    "weight",
    ascending=False
)

weight_table.to_csv(
    "entropy_feasibility_weightage.csv",
    index=False
)

# =====================================================
# Feature Contributions
# =====================================================

for col in criteria:

    df[col+"_normalized"] = normalized[col]

    df[col+"_contribution"] = normalized[col] * weights[col]

# =====================================================
# Final Feasibility Score
# =====================================================

contribution_cols = [c+"_contribution" for c in criteria]

df["feasibility_score"] = (
    df[contribution_cols]
      .sum(axis=1)
      * 100
)

# =====================================================
# KMeans Clustering
# =====================================================

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=20
)

df["cluster"] = kmeans.fit_predict(
    df[["feasibility_score"]]
)

# =====================================================
# Assign Labels
# =====================================================

cluster_order = (
    df.groupby("cluster")["feasibility_score"]
      .mean()
      .sort_values()
      .index
)

label_map = {
    cluster_order[0]: "Low",
    cluster_order[1]: "Moderate",
    cluster_order[2]: "High"
}

df["feasibility_label"] = df["cluster"].map(label_map)

# =====================================================
# Save Final Dataset
# =====================================================

df.to_csv(
    "dataset_labelled_with_score.csv",
    index=False
)

# =====================================================
# Display Results
# =====================================================

print("\nEntropy Weights\n")
print(weight_table)

print("\nCluster Means\n")
print(
    df.groupby("cluster")["feasibility_score"]
      .mean()
      .sort_values()
)

print("\nLabel Distribution\n")
print(df["feasibility_label"].value_counts())

print("\nFiles Saved Successfully")
print("1. entropy_feasibility_weightage.csv")
print("2. dataset_labelled_with_score.csv")


Entropy Weights

                       feature criterion_type   entropy  divergence    weight
9            cinema_count_500m        Benefit  0.853503    0.146497  0.176918
10           museum_count_500m        Benefit  0.867034    0.132966  0.160577
5          bus_stop_count_500m        Benefit  0.928183    0.071817  0.086730
2     parking_space_count_500m        Benefit  0.938637    0.061363  0.074106
11           temple_count_500m        Benefit  0.942178    0.057822  0.069830
6           college_count_500m        Benefit  0.942649    0.057351  0.069261
4              bank_count_500m        Benefit  0.942894    0.057106  0.068965
13        nearest_restaurant_m        Benefit  0.954186    0.045814  0.055327
12           clinic_count_500m        Benefit  0.961915    0.038085  0.045993
0            office_count_500m        Benefit  0.965170    0.034830  0.042063
7          hospital_count_500m        Benefit  0.965594    0.034406  0.041551
3        recreation_count_500m        Benefit 